# Get Patch-Level Labels for TCGA-HNSCC Dataset

For this dataset, we folllowed two-steps processing to get the patch-level lables:
1. use the [CellViT++](https://github.com/TIO-IKIM/CellViT-plus-plus) to detect the cells information(position and cell type). Then aggregate the cell information for each patch.
2. Following the tissue segmentation in [Digital Spatial Pathway Mapping Reveals Prognostic Tumor States in Head and Neck Cancer](https://github.com/bifold-pathomics/xMIL-Pathways/tree/main) to get the tumor, non-tumor, and border segmentation. Since the border has also tumor signal, so we merge it with the tumor type. then we will only focus on the following tissue types: **'Tumor-Neoplastic', 'Tumor-Inflammatory', 'Non-Tumor-Inflammatory', 'Non-Tumor-Connective', 'Tumor-Connective'**


## Aggregate the Cell level information to patch level

In [ ]:
import os
import json
import ast
import pandas as pd

def flatten_geojson_cells(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, dict) and "features" in data:
        features = data["features"]
    elif isinstance(data, list):
        features = data
    else:
        raise ValueError("Unrecognized GeoJSON format")

    rows = []
    for feature in features:
        coords = feature.get("geometry", {}).get("coordinates", [])
        cell_type = feature.get("properties", {}).get("classification", {}).get("name", "Unknown")
        for coord in coords:
            rows.append({"x": coord[0], "y": coord[1], "classification": cell_type})
    return pd.DataFrame(rows)

def get_patch_label(patch_row, df_cellvit):
    x_abs, y_abs = patch_row['position_abs']
    patch_size = patch_row['patch_size_abs']
    x_min = x_abs - (patch_size / 2)
    y_min = y_abs - (patch_size / 2)
    x_max = x_abs + (patch_size / 2)
    y_max = y_abs + (patch_size / 2)
    inside_cells = df_cellvit[
        (df_cellvit['x'] >= x_min) & (df_cellvit['x'] <= x_max) &
        (df_cellvit['y'] >= y_min) & (df_cellvit['y'] <= y_max)
    ]
    default_counts = {
        "Neoplastic": 0,
        "Connective": 0,
        "Inflammatory": 0,
        "Epithelial": 0,
        "Dead": 0
    }
    if not inside_cells.empty:
        actual_counts = inside_cells["classification"].value_counts().to_dict()
        default_counts.update(actual_counts)

    return json.dumps(default_counts)

def get_major_label(annotation_str):
    """
    Assign one patch-level label using majority voting over CellViT++ cell counts.

    Parameters
    ----------
    annotation_cellvit : str or dict
        CellViT++ class-count annotation for one patch.

    Returns
    -------
    str
        Majority class label. Returns 'Empty' when the annotation is missing,
        empty, or contains only zero counts.
    """
    try:
        ann = ast.literal_eval(annotation_str)
        if isinstance(ann, dict) and ann:
            return max(ann.items(), key=lambda x: x[1])[0]  # get label with max count
        else:
            return "Empty"
    except:
        return "Invalid"

def label_patches_by_cell_geojson(patch_metadata_dir, cellvit_geojson_dir, out_dir):
    """
    Assign CellViT++-based patch labels to each patch and save updated patch metadata.

    Parameters
    ----------
    patch_metadata_dir : 
        Directory containing patch metadata CSV files for all WSIs. Each CSV must contain `patch_id` and patch location information.

    cellvit_geojson_dir : 
        Directory containing CellViT++ cell detection GeoJSON files for all WSIs. Each file

    out_dir :
        Output directory where labeled patch metadata CSV files are saved. 

    Returns 
    -------
    Writes CSV files containing patch metadata with CellViT++-based patch labels.

    """

    print(f"Reading WSI list from {patch_metadata_dir}...", flush=True)
    wsi_list = os.listdir(patch_metadata_dir)
    print(f"Found {len(wsi_list)} WSI folders", flush=True)
    total = len(wsi_list)

    for idx, wsi_name in enumerate(wsi_list, start=1):
        print(f"Processing WSI {idx}/{total}: {wsi_name}", flush=True)

        output_csv = os.path.join(out_dir, wsi_name, 'metadata', 'df.csv')
        if os.path.exists(output_csv):
            print(f"Already labeled: {wsi_name}, skipping.")
            continue

        patch_csv = os.path.join(patch_metadata_dir, wsi_name, 'metadata', 'df.csv')

        try:
            patch_metadata = pd.read_csv(patch_csv)
            patch_metadata['position_abs'] = patch_metadata['position_abs'].apply(ast.literal_eval)

            geojson_path = os.path.join(cellvit_geojson_dir, f'{wsi_name}_cell_detection.geojson')
            if not os.path.exists(geojson_path):
                print(f"GeoJSON not found for {wsi_name}, skipping.")
                continue

            df_cellvit = flatten_geojson_cells(geojson_path)
            patch_metadata['annotation_cellvit'] = patch_metadata.apply(
                lambda row: get_patch_label(row, df_cellvit), axis=1
            )
            os.makedirs(os.path.dirname(output_csv), exist_ok=True)
            patch_metadata.to_csv(output_csv, index=False)
            print(f"Labeled metadata saved to: {output_csv}")

        except Exception as e:
            print(f"Error processing {wsi_name}: {e}")


def label_one_slide_by_cell_geojson(wsi_name, patch_metadata_dir, cellvit_dir, out_dir):
    patch_csv = os.path.join(patch_metadata_dir, wsi_name, 'metadata', 'df.csv')

    try:
        patch_metadata = pd.read_csv(patch_csv)
        patch_metadata['position_abs'] = patch_metadata['position_abs'].apply(ast.literal_eval)
        # Get GeoJSON path
        cellvit_geojson = os.path.join(cellvit_dir, f'{wsi_name}_cell_detection.geojson')
        # Label patches
        df_cellvit = flatten_geojson_cells(cellvit_geojson)
        patch_metadata['annotation_cellvit'] = patch_metadata.apply(
            lambda row: get_patch_label(row, df_cellvit), axis=1
        )
        print(f"Patch labeling complete for {wsi_name}")

        # Save output
        output_csv = os.path.join(out_dir, wsi_name, 'metadata', 'df.csv')
        os.makedirs(os.path.dirname(output_csv), exist_ok=True)
        patch_metadata.to_csv(output_csv, index=False)
        print(f"Saved labeled metadata to: {output_csv}")

    except Exception as e:
        print(f"Error: {e}")


#Example usage
# get the label for all slides in a dataset
label_patches_by_cell_geojson(
    patch_metadata_dir='/path/to/patch_metadata', 
    cellvit_geojson_dir='/path/to/cellvit_geojson', 
    out_dir='/path/to/output'
)
# only get the label for one specific slide
label_one_slide_by_cell_geojson(
    wsi_name='wsi_example', 
    patch_metadata_dir='/path/to/patch_metadata', 
    cellvit_dir='/path/to/cellvit', 
    out_dir='/path/to/output'
)

# aggregate the tissue segmentation information with cellvit file

In [ ]:
df_tissue = pd.read_csv("/path/to/tissue_based_patch_metadata.csv")
# columns: 'slide_id', 'patch_id', 'tissue_label'

wsi_folder = "/path/to/wsi_folder"

# Mapping from tissue_label to border_score
TISSUE_SCORE_MAP = {
    "Border": 1,
    "Tumor": 2,
    "Non-Tumor": 0,
}

for slide_id in os.listdir(wsi_folder):
    # please adjust this based on your actual folder structure and naming convention
    csv_path = os.path.join(wsi_folder, slide_id, "metadata/df.csv")
    if not os.path.exists(csv_path):
        continue

    df_cellvit = pd.read_csv(csv_path)
    # columns: 'slide_id', 'patch_id', 'annotation_cellvit'

    df_tissue_slide = df_tissue[df_tissue["slide_id"] == slide_id][["patch_id", "tissue_label"]]

    # Merge tissue label in
    df_cellvit = df_cellvit.merge(df_tissue_slide, on="patch_id", how="left")

    # Map tissue_label -> border_score
    df_cellvit["border_score"] = df_cellvit["tissue_label"].map(TISSUE_SCORE_MAP)

    # Drop the intermediate tissue_label column if you don't need it
    df_cellvit.drop(columns=["tissue_label"], inplace=True)

    # Overwrite the original CSV with the new border_score column
    df_cellvit.to_csv(csv_path, index=False)

    print(f"Updated: {csv_path} — {len(df_cellvit)} patches")

### Analysis of Patch-level Label proportion

In [ ]:
import sys
sys.path.insert(0, "./models/xMIL") # please update this path to your dataset handler module path

if "datasets" in sys.modules:
    del sys.modules["datasets"]

from datasets.data_handler import SlideDataHandler

# Update this to your CellViT++-labeled patch metadata directory. Our dataset structure is /wsi_folde/slide_id/metadata/df.csv
wsi_metadata_folder = "/path/to/your/cellvit_labeled_metadata" 

rows = []

classes = [
    "Tumor-Neoplastic",
    "Tumor-Inflammatory",
    "Tumor-Connective",
    "Non-Tumor-Inflammatory",
    "Non-Tumor-Connective"
]

for wsi in os.listdir(wsi_metadata_folder):

    wsi_path = os.path.join(wsi_metadata_folder, wsi)

    if not os.path.isdir(wsi_path):
        continue

    csv_path = os.path.join(wsi_path, "metadata", "df.csv")

    if not os.path.exists(csv_path):
        continue

    df_csv = pd.read_csv(csv_path)

    df_csv["patch_label"] = SlideDataHandler.get_patch_labels(
        df_csv,
        "hnsc_majority_border"
    )

    # proportions
    proportions = df_csv["patch_label"].value_counts(normalize=True)

    row = {"slide_id": wsi}

    for c in classes:
        row[c] = proportions.get(c, 0.0)

    rows.append(row)


df_proportion = pd.DataFrame(rows)

print(df_proportion.head())

df_proportion.to_csv(
    "path/to/save/hnscc_patch_label_proportions.csv",
    index=False
)

### Visualization of the patch_label_proportion

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# -----------------------------
# Load positive slides
# -----------------------------
df_positive = pd.read_csv(
    "/path/to/your/positive_slides_list.csv"
)

df_proportion["slide_id"] = df_proportion["slide_id"].astype(str).str.strip()

df_positive["slide_id"] = (
    df_positive["slide_id"]
    .astype(str)
    .str.strip()
    .str.strip("[]")
    .str.strip("'")
    .str.strip('"')
)

positive_ids = set(df_positive["slide_id"])

classes = [
    "Tumor-Neoplastic",
    "Tumor-Inflammatory",
    "Tumor-Connective",
    "Non-Tumor-Inflammatory",
    "Non-Tumor-Connective",
]

palette = {
    "T-Neo": "#d73027",   # dark red

    "T-Inf": "#1b7837",   # dark green
    "T-Con": "#2166ac",   # dark blue

    "NT-Inf": "#a6dba0",  # light green
    "NT-Con": "#92c5de",  # light blue
}

# -----------------------------
# Prepare long-form data
# -----------------------------
df_all_long = df_proportion.melt(
    id_vars="slide_id",
    value_vars=classes,
    var_name="Patch type",
    value_name="Proportion"
)

df_pos_long = df_proportion[
    df_proportion["slide_id"].isin(positive_ids)
].melt(
    id_vars="slide_id",
    value_vars=classes,
    var_name="Patch type",
    value_name="Proportion"
)

label_map = {
    "Tumor-Neoplastic": "T-Neo",
    "Tumor-Inflammatory": "T-Inf",
    "Tumor-Connective": "T-Con",
    "Non-Tumor-Inflammatory": "NT-Inf",
    "Non-Tumor-Connective": "NT-Con",
}

df_all_long["Patch type"] = df_all_long["Patch type"].map(label_map)
df_pos_long["Patch type"] = df_pos_long["Patch type"].map(label_map)

order = ["T-Neo", "T-Inf", "T-Con", "NT-Inf", "NT-Con"]

# -----------------------------
# Style
# -----------------------------
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.8,
})

sns.set_style("whitegrid")

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.0, 2.8),
    sharey=True
)

for ax, data, title in zip(
    axes,
    [df_all_long, df_pos_long],
    ["All slides", "Positive slides"]
):
    sns.boxplot(
        data=data,
        x="Patch type",
        y="Proportion",
        order=order,
        ax=ax,
        palette=palette,
        width=0.55,
        showfliers=False,
        linewidth=1.0,
        boxprops={"edgecolor":"black"},
        whiskerprops={"color":"black"},
        capprops={"color":"black"},
        medianprops={"color":"black","linewidth":1.2},
    )

    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylim(0, 1.02)
    ax.tick_params(axis="x", rotation=25)

    ax.grid(axis="y", linewidth=0.5, alpha=0.6)
    ax.grid(axis="x", visible=False)

axes[0].set_ylabel("Patch proportion")
axes[1].set_ylabel("")

sns.despine(fig=fig, top=True, right=True)

fig.tight_layout(w_pad=1.2)

plt.savefig(
    "/path/to/save/hnsc_patch_proportion_overview_positive_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

![figure](hnsc_patch_proportion_overview_positive_boxplot.png)